## Model Interpretation only (due to cacheing issue in Databricks Free Tier)

This notebook focuses on:
- model interpretation (what features drive predictions)
- report-ready summary tables

#### Load up stored data

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS
from pyspark.ml.functions import vector_to_array

FINAL_SELECTION_TABLE = "workspace.bda_taxi.model_comparison_final"

final_selection = spark.table(FINAL_SELECTION_TABLE)

display(final_selection)

## Model interpretation (refitting best model only)

Refitting the best model that was stored in a table in previous notebook.
Depending on the model that it is:
- Logistic Regression: coefficients
- Random Forest: feature importances

##### Reuse the pipeline builders and feature lists from shared notebook 04

In [0]:
# COMMAND ----------
import math
import gc
from typing import List, Tuple

from pyspark.sql import functions as SQL_FUNCTIONS
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel
from pyspark.ml.feature import StringIndexerModel, OneHotEncoderModel, VectorAssembler, StandardScalerModel

# COMMAND ----------
FINAL_SELECTION_TABLE = "workspace.bda_taxi.model_comparison_final"
TRAIN_TABLE = "workspace.bda_taxi.model_train_set"

# Output tables
LR_COEF_FULL_TABLE = "workspace.bda_taxi.logreg_coefficients_full"
LR_COEF_TOP_POS_TABLE = "workspace.bda_taxi.logreg_coefficients_top_positive"
LR_COEF_TOP_NEG_TABLE = "workspace.bda_taxi.logreg_coefficients_top_negative"

RF_IMPORTANCE_TABLE = "workspace.bda_taxi.random_forest_feature_importance"

TOP_N = 50
print("FINAL_SELECTION_TABLE:", FINAL_SELECTION_TABLE)
print("TRAIN_TABLE:", TRAIN_TABLE)

In [0]:
# COMMAND ----------
final_selection_df = spark.table(FINAL_SELECTION_TABLE)
best_model_name = final_selection_df.select("model").first()["model"]
print("Best model:", best_model_name)


In [0]:
%run ./04_model_training_and_evaluation_shared

In [0]:

train_df = spark.table(TRAIN_TABLE)
print("Train rows:", train_df.count())

In [0]:

def find_stage(model: PipelineModel, stage_type):
    for s in model.stages:
        if isinstance(s, stage_type):
            return s
    return None

def find_all_stages(model: PipelineModel, stage_type):
    return [s for s in model.stages if isinstance(s, stage_type)]

def write_overwrite_table(df, table_name: str) -> None:
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(table_name)
    )
    print("Saved:", table_name)

In [0]:
def find_stage(model: PipelineModel, stage_type):
    for s in model.stages:
        if isinstance(s, stage_type):
            return s
    return None

def find_all_stages(model: PipelineModel, stage_type):
    return [s for s in model.stages if isinstance(s, stage_type)]

def write_overwrite_table(df, table_name: str) -> None:
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(table_name)
    )
    print("Saved:", table_name)

##### Specific Logisitic regression interpretation function

In [0]:

def interpret_logistic_regression() -> None:
    """
    Fits the LR pipeline once, extracts coefficients, maps them to feature names,
    and writes full + top pos/neg tables.

    NOTE: Full readable categorical names require OneHotEncoder to include ALL categorical idx cols
    in a single OneHotEncoder stage (recommended fix in build_preprocessing_stages()).
    """
    lr_pipeline = build_logistic_regression_pipeline(
        numeric_cols=numeric_features,
        categorical_cols=categorical_features,
        label_column=label_col
    )

    fitted: PipelineModel = lr_pipeline.fit(train_df)
    lr_model: LogisticRegressionModel = find_stage(fitted, LogisticRegressionModel)
    if lr_model is None:
        raise ValueError("Could not find LogisticRegressionModel in fitted pipeline")

    # Locate preprocessing stages
    indexer_models: List[StringIndexerModel] = find_all_stages(fitted, StringIndexerModel)
    encoder_model: OneHotEncoderModel = find_stage(fitted, OneHotEncoderModel)

    coef_array = lr_model.coefficients.toArray()

    # ----- Build feature names -----
    # Numeric first
    feature_names = list(numeric_features)

    # Categorical OHE names (only reliable if encoder_model contains all idx cols)
    if encoder_model is not None:
        labels_by_indexed_output = {m.getOutputCol(): list(m.labels) for m in indexer_models}
        enc_input_cols = list(encoder_model.getInputCols())
        enc_sizes = list(encoder_model.categorySizes)

        # Create OHE names in the correct order
        for idx_col, size in zip(enc_input_cols, enc_sizes):
            base_col = idx_col.replace("_idx", "")
            labels = labels_by_indexed_output.get(idx_col, [])

            # pad if needed for handleInvalid="keep"
            while len(labels) < size:
                labels.append("__unknown__")

            for j in range(size):
                feature_names.append(f"{base_col}={labels[j]}")
    else:
        # If encoder stage not found, fall back to indexed placeholder naming
        # (This should not happen if using OHE)
        missing = len(coef_array) - len(feature_names)
        for j in range(missing):
            feature_names.append(f"cat_feature_{j}")

    if len(feature_names) != len(coef_array):
        raise ValueError(
            f"Feature name count ({len(feature_names)}) != coefficient count ({len(coef_array)}). "
            "If LR naming is required, ensure OneHotEncoder encodes ALL categorical columns in one stage."
        )

    # ----- Build coefficient dataframe -----
    rows = []
    for name, coef in zip(feature_names, coef_array):
        coef_f = float(coef)
        rows.append((name, coef_f, abs(coef_f), float(math.exp(coef_f))))

    coef_df = spark.createDataFrame(rows, ["feature", "coef", "abs_coef", "odds_ratio"])
    coef_df_sorted = coef_df.orderBy(SQL_FUNCTIONS.desc("abs_coef"))

    write_overwrite_table(coef_df_sorted, LR_COEF_FULL_TABLE)

    top_pos = coef_df.orderBy(SQL_FUNCTIONS.desc("coef")).limit(TOP_N)
    top_neg = coef_df.orderBy(SQL_FUNCTIONS.asc("coef")).limit(TOP_N)

    write_overwrite_table(top_pos, LR_COEF_TOP_POS_TABLE)
    write_overwrite_table(top_neg, LR_COEF_TOP_NEG_TABLE)

    # Cleanup and exit
    del fitted
    del lr_pipeline
    gc.collect()

##### Specific Random forest interpretation function

In [0]:

def interpret_random_forest() -> None:
    """
    Fits the RF pipeline once, extracts feature importances and saves them.

    Note:
    - With OHE features, importances are at the expanded feature level.
    - If you want importances at the original feature level, use an indexed-categorical interpretability RF
      (requires maxBins>=max category cardinality).
    """
    rf_pipeline = build_random_forest_pipeline(
        numeric_cols=numeric_features,
        categorical_cols=categorical_features,
        label_column=label_col
    )

    fitted: PipelineModel = rf_pipeline.fit(train_df)
    rf_model: RandomForestClassificationModel = find_stage(fitted, RandomForestClassificationModel)
    if rf_model is None:
        raise ValueError("Could not find RandomForestClassificationModel in fitted pipeline")

    importances = rf_model.featureImportances.toArray()
    rows = [(int(i), float(v)) for i, v in enumerate(importances)]

    importance_df = spark.createDataFrame(rows, ["feature_index", "importance"]) \
                         .orderBy(SQL_FUNCTIONS.desc("importance"))

    write_overwrite_table(importance_df, RF_IMPORTANCE_TABLE)

    del fitted
    del rf_pipeline
    gc.collect()

#### Based on best model, run the associated interpretation function

In [0]:

if best_model_name == "LogisticRegression":
    print("Running Logistic Regression interpretation...")
    interpret_logistic_regression()
    dbutils.notebook.exit("LR interpretation complete - exiting.")
elif best_model_name == "RandomForest":
    print("Running Random Forest interpretation...")
    interpret_random_forest()
    dbutils.notebook.exit("RF interpretation complete - exiting.")
else:
    dbutils.notebook.exit(f"Unknown model: {best_model_name}. Nothing to interpret.")